# COde 01 — Features only: original or flattened Qwen images
Attach ClinDistill bank + prepared release (image_index.json) + code_full_offline_assets. GPU ON, Internet OFF, Run All. No re-splitting or raw CSV audit.


In [ ]:
from pathlib import Path
import os, sys, json, importlib.util
RESEARCH_BANK_OVERRIDE = globals().get('RESEARCH_BANK_OVERRIDE')
RESEARCH_ASSETS = None

def research_find(predicate, *, include_runs=False):
    found = set()
    for base in (Path.cwd(), Path.cwd().parent / 'Data', Path('/kaggle/input')):
        if not base.exists():
            continue
        for current, dirs, files in os.walk(base):
            here = Path(current)
            if predicate(here, dirs, files):
                found.add(here.resolve())
            excluded = {'Images', 'images', '.git', '.cache', '__pycache__', '_archive', '_support',
                        'models', 'wheels', 'wheelhouse', 'official_code', '_chunks', 'student_export'}
            dirs[:] = [d for d in dirs if not d.lower().startswith('qwen')]
            if not include_runs:
                excluded.add('runs')
            dirs[:] = [d for d in dirs if d not in excluded]
            if len(here.relative_to(base).parts) >= 8:
                dirs[:] = []
    return sorted(found, key=str)

def research_one(override, found, name):
    if override is not None:
        return Path(override).resolve()
    if len(found) != 1:
        raise RuntimeError(f'{name}: found {len(found)} folders: {found}. Set the matching *_OVERRIDE.')
    return found[0]

if Path('/kaggle').exists():
    bank_root = research_one(RESEARCH_BANK_OVERRIDE,
        research_find(lambda p, d, f: 'research_bank_manifest.json' in f), 'Research bank')
    helper = bank_root / 'source/_support/research_bank.py'
    spec = importlib.util.spec_from_file_location('research_bank_setup', helper)
    bank = importlib.util.module_from_spec(spec); spec.loader.exec_module(bank)
    RESEARCH_ASSETS = bank.activate(bank_root)
    PROJECT_ROOT = RESEARCH_ASSETS['source_root']
    # Compatibility for existing data-audit publishing cells only.
    OFFLINE_ASSETS = {'root': bank_root, 'source_root': PROJECT_ROOT}
else:
    PROJECT_ROOT = research_one(globals().get('PROJECT_ROOT_OVERRIDE'),
        research_find(lambda p, d, f: (p / 'clindistill/research').is_dir() and 'configs' in d), 'Research source')
    sys.path.insert(0, str(PROJECT_ROOT))
print('Research source:', PROJECT_ROOT)


In [ ]:
from clindistill.artifacts import read_json
from clindistill.research.data import prepare_features
PREPARED_ROOT_OVERRIDE = None
IMAGE_ROOT_OVERRIDE = None
RESEARCH_FEATURE_OUTPUT_OVERRIDE = None
MODEL_LOCATIONS_OVERRIDE = None
IMAGE_BATCH_SIZE = 'auto'  # >=70 GiB free: B=256, L=128, ResNet=256, Swin=128
IMAGE_WORKERS = 'auto'  # up to 8 CPU workers; CPU software checks use 0
CLINICAL_BATCH_SIZE = 'auto'  # 128 on a largely free 96-GB GPU; FP32 text encoder
TEST_PROTOCOL_PATH = None
PREPARE_SPLITS = ['train', 'validation']
prepared = research_one(PREPARED_ROOT_OVERRIDE,
    [release_root] if 'release_root' in globals() else research_find(lambda p,d,f:
        'release.json' in f and not read_json(p / 'release.json').get('debug_subset')), 'Prepared release')
images = research_one(IMAGE_ROOT_OVERRIDE, research_find(lambda p,d,f:
    ((p / 'Images/Photographs').is_dir() or ('images' in d and 'cls_train.json' in f))
    and 'mini_dataset.json' not in f), 'Original CODe or Qwen data/images')
models = MODEL_LOCATIONS_OVERRIDE or (RESEARCH_ASSETS['models'] if RESEARCH_ASSETS else None)
if models is None:
    raise RuntimeError('Attach the ClinDistill research bank; Qwen weights are not DINO/BGE assets.')
cfg = read_json(PROJECT_ROOT / 'configs/code/research.json')
cfg.update(image_batch_size=IMAGE_BATCH_SIZE, image_workers=IMAGE_WORKERS, clinical_batch_size=CLINICAL_BATCH_SIZE)
destination = Path(RESEARCH_FEATURE_OUTPUT_OVERRIDE) if RESEARCH_FEATURE_OUTPUT_OVERRIDE else (
    Path('/kaggle/working/data/COde_processed/research_features_v3') if Path('/kaggle').exists()
    else PROJECT_ROOT.parent / 'Data/COde_processed/research_features_v3')
test_protocol = read_json(TEST_PROTOCOL_PATH) if TEST_PROTOCOL_PATH else None
def verified_image_view(image_roots, image_index, workspace, *, required_paths=None):
    """Create audited filesystem aliases usable by the installed 0.5.1 wheel.

    Only byte-identical photos are admitted. The canonical root is the common
    ancestor of inputs and working storage, keeping symlink targets inside the
    existing wheel's path boundary. No installed functions or manifests change.
    """
    from pathlib import Path, PurePosixPath
    import hashlib
    import json
    import os

    roots = [Path(p).resolve() for p in image_roots]
    work = Path(workspace).resolve()
    if not roots or not image_index:
        raise ValueError('Image roots and original image_index.json are required.')
    required = set() if required_paths is None else set(required_paths)
    if not required <= set(image_index):
        raise ValueError('Required photos are absent from the original image index.')
    for name, digest in image_index.items():
        parts = PurePosixPath(name).parts
        if (len(parts) != 3 or parts[:2] != ('Images', 'Photographs')
                or '\\' in name or '..' in parts or len(digest) != 64
                or any(c not in '0123456789abcdef' for c in digest)):
            raise ValueError('Invalid canonical photo path or SHA256 in image index.')
    if all((roots[0] / name).is_file() for name in (required or image_index)):
        return roots[0]  # Original CODe tree: prepare_features performs the hash verification.

    def sha256(path):
        h = hashlib.sha256()
        with path.open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                h.update(block)
        return h.hexdigest()

    common = Path(os.path.commonpath([str(p) for p in roots] + [str(work)]))
    if common == Path(common.anchor):
        raise ValueError('Inputs and workspace need a shared parent below the filesystem root, e.g. /kaggle.')
    work.mkdir(parents=True, exist_ok=True)
    report_path = work / 'image_layout_report.json'
    mapping, missing, mismatched, hashes = {}, [], [], {}
    # Map all available indexed photos, so adding a locked test shard needs no new
    # layout policy. Strict coverage applies to the requested train/validation set.
    for number, name in enumerate(sorted(image_index), 1):
        packaged = hashlib.sha1(name.encode('utf-8')).hexdigest()[:10] + '_' + PurePosixPath(name).name
        candidates = []
        for root in roots:
            for candidate in (root / name, root / 'images' / packaged, root / packaged,
                              root / 'images' / PurePosixPath(name).name, root / PurePosixPath(name).name):
                if not candidate.is_file():
                    continue
                source = candidate.resolve()
                if root not in source.parents or common not in source.parents:
                    raise ValueError('Packaged image escapes the selected input root.')
                if source not in candidates:
                    candidates.append(source)
        selected = None
        for source in candidates:
            if source not in hashes:
                hashes[source] = sha256(source)
            if hashes[source] == image_index[name]:
                selected = source
                break
        if selected is None:
            (mismatched if candidates else missing).append(name)
        else:
            mapping[name] = str(selected)
        if number % 2000 == 0:
            print(f'Image layout: verified {number:,}/{len(image_index):,} canonical photos', flush=True)

    absent = sorted(required - set(mapping))
    proof = hashlib.sha256(json.dumps(image_index, sort_keys=True).encode()).hexdigest()
    report = {'schema_version': 1, 'policy': 'sha1(canonical_path)[:10]_basename; original SHA256 required',
              'image_index_identity': proof, 'source_roots': list(map(str, roots)),
              'canonical_root': str(common), 'indexed': len(image_index), 'mapped': len(mapping),
              'required': len(required), 'missing_required': absent,
              'unavailable_indexed': missing, 'mismatched_indexed': mismatched, 'mapping': mapping,
              'status': 'missing_required' if absent else 'coverage_verified'}
    report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    if absent:
        raise FileNotFoundError(f'{len(absent)} required photos are missing or differ from original bytes. '
                                f'First: {absent[0]}. Attach the CODe image supplement or original photos. '
                                f'Coverage report: {report_path}')

    anchor = common / 'Images'
    marker = anchor / '.clindistill-image-layout.json'
    if anchor.exists():
        if anchor.is_symlink() or not marker.is_file():
            raise FileExistsError('Canonical Images directory already exists and is not managed by this adapter.')
        info = json.loads(marker.read_text(encoding='utf-8'))
        if info.get('image_index_identity') != proof:
            raise ValueError('Existing canonical image view belongs to another image index. Use a fresh session.')
    else:
        anchor.mkdir()
        marker.write_text(json.dumps({'image_index_identity': proof}) + '\n', encoding='utf-8')

    counts = {'symlink': 0, 'hardlink': 0, 'reused': 0}
    for name, source in mapping.items():
        dest, source = common / name, Path(source)
        dest.parent.mkdir(parents=True, exist_ok=True)
        if dest.is_symlink() or dest.exists():
            if not dest.is_file() or not os.path.samefile(dest, source):
                raise FileExistsError('Existing canonical alias points to another image; use a fresh session.')
            counts['reused'] += 1
            continue
        try:
            dest.symlink_to(source)
            counts['symlink'] += 1
        except OSError:
            # Windows regression checks can use hardlinks on the same volume.
            # On Kaggle, symlinks connect read-only Input files to the canonical view.
            os.link(source, dest)
            counts['hardlink'] += 1
    report['aliases'] = counts
    report['status'] = 'verified'
    report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print(f'IMAGE LAYOUT READY: {len(mapping):,} byte-verified photos; root={common}; report={report_path}', flush=True)
    return common
if (prepared / 'image_index.json').is_file():
    from clindistill.artifacts import verify_release
    from clindistill.data import load_records, deterministic_paths
    image_index = read_json(prepared / 'image_index.json')
    if image_index['release_id'] != read_json(prepared / 'release.json')['release_id']:
        raise ValueError('Image index belongs to another prepared release.')
    extra_roots = research_find(lambda p,d,f: 'code_image_supplement.json' in f)
    for extra in extra_roots:
        if read_json(extra / 'code_image_supplement.json')['release_id'] != image_index['release_id']:
            raise ValueError('Image supplement belongs to another release.')
    required = None
    if 'test' not in PREPARE_SPLITS:
        verify_release(prepared, splits=PREPARE_SPLITS)
        required = {p for split in PREPARE_SPLITS
            for row in load_records(prepared, split, 'photo_attention')
            for p in deterministic_paths(row['photo_paths'], cfg['max_photos'])}
    # This creates byte-verified filesystem aliases only. The installed wheel
    # still enforces the locked-test protocol before extracting any features.
    images = verified_image_view([images, *extra_roots], image_index['images'],
                                 destination.parent / 'image_layout', required_paths=required)
manifest = prepare_features(prepared, images, models, destination, cfg, splits=PREPARE_SPLITS, test_protocol=test_protocol)
print('Research cache:', destination, '\nDimensions:', manifest['dimensions'], '\nSplits:', manifest['splits'])
print('Save Version and attach this Output to notebook02. Runtime: feature_runtime.json')
